# Datenaufbereitung $SpO_2$ (Lösung)

In diesem Notebook schauen wir uns die Daten genauer an, mit denen wir arbeiten werden. Anschließend bereiten wir sie so auf, dass wir sie in der nächsten Sitzung als Eingabe für unser Machine-Learning-Modell mit scikit-learn verwenden können.

### Der Datensatz
Die Datei `data_spo2.csv` enthält Messungen der **peripheren Sauerstoffsättigung ($SpO_2$)** von Patient\*innen auf der Intensivstation. Jede Zeile ist eine einzelne Messung.

| Spalte | Bedeutung |
|---|---|
| `index` | gespeicherte Zeilennummer |
| `subject_id` | Pseudonym der Patientin / des Patienten |
| `admission_id` | Krankenhausaufenthalt |
| `stay_id` | Aufenthalt auf der Intensivstation (eine Person kann mehrere haben) |
| `time` | Zeitpunkt der Messung |
| `spo2` | Sauerstoffsättigung in % |

> **Warum liegen die Daten im Jahr 2180?** Die Daten stammen aus einer anonymisierten Intensivdatenbank. Zum Schutz der Patient\*innen wurden alle Datumsangaben pro Person zufällig in die Zukunft verschoben. Die Abstände zwischen den Messungen bleiben dabei erhalten.

### Hinweise
- Legen Sie die Datei `data_spo2.csv` in denselben Ordner wie dieses Notebook.
- Alles, was Sie brauchen, kennen Sie aus den Tutorials 02 (Pandas) und 03 (Matplotlib). Bei jeder Aufgabe steht ein Tipp, wo Sie nachschauen können.

# Datenexploration

## Pakete importieren

Alle benötigten Pakete werden zu Beginn des Notebooks importiert.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Aufgabe 1: Daten laden

Laden Sie die CSV-Datei als DataFrame und sehen Sie sich die ersten 5 Zeilen an.

*Tipp:* Die erste Spalte `index` ist nur die gespeicherte Zeilennummer. Mit `pd.read_csv(..., index_col=0)` wird sie als Index des DataFrames verwendet statt als zusätzliche Datenspalte. (Tutorial 02, Abschnitt 2 und 3)

In [ ]:
df = pd.read_csv("02-5_data_spo2.csv", index_col=0)
df.head()

## Aufgabe 2: Datentypen prüfen

Prüfen Sie die Datentypen der Spalten. Passen alle zum Inhalt der Spalte?

*Tipp:* `df.dtypes` oder `df.info()`. Zeitangaben sollten den Datentyp `datetime64` haben, damit man mit ihnen rechnen und sie korrekt plotten kann. Die Umwandlung gelingt mit `pd.to_datetime(...)`.

In [ ]:
df.dtypes

Die Spalte `time` ist als Text gespeichert. Wir wandeln sie in einen echten Zeitstempel um:

In [ ]:
df["time"] = pd.to_datetime(df["time"])
df.dtypes

Die ID-Spalten sind ganze Zahlen (`int64`). Das ist in Ordnung, man sollte aber im Kopf behalten, dass es sich um **Bezeichner** handelt und nicht um Messgrößen: Ein Mittelwert der `subject_id` ergibt keinen Sinn.

## Aufgabe 3: Nicht benötigte Spalte entfernen

Die `admission_id` brauchen wir für diese Aufgabe nicht. Entfernen Sie die Spalte.

*Tipp:* `df.drop(columns=...)`. Denken Sie daran, das Ergebnis wieder `df` zuzuweisen. (Tutorial 02, Abschnitt 7)

In [ ]:
df = df.drop(columns="admission_id")
df.head()

## Aufgabe 4: Fehlende Werte prüfen

$SpO_2$ ist unsere Messgröße von Interesse. Prüfen Sie, ob die Spalte `spo2` fehlende Werte (`NaN`) enthält.

*Tipp:* `.isna().sum()` (Tutorial 02, Abschnitt 7)

In [ ]:
print("Fehlende SpO2-Werte:", df["spo2"].isna().sum())

> **Zum Nachdenken:** Fehlende Werte gibt es keine. Heißt das, alle Werte sind *plausibel*? Schauen Sie sich bei Aufgabe 6 das Minimum an. Woher könnten sehr niedrige Werte kommen (echte Hypoxie, verrutschter Sensor, kalte Finger …)? Wir verändern den Datensatz hier bewusst nicht, aber in einem echten Projekt müsste man solche Werte prüfen.

## Aufgabe 5: Fertigen Datensatz benennen

Die Vorverarbeitung ist abgeschlossen. Speichern Sie Ihren DataFrame unter dem Namen `final_df`. Mit diesem Datensatz arbeiten wir in der nächsten Sitzung weiter.

In [ ]:
final_df = df
print(final_df.shape)
final_df.head()

## Datensatz speichern

Damit wir den aufbereiteten Datensatz in der nächsten Sitzung (Modellentwicklung) laden können, speichern wir ihn als CSV-Datei. Führen Sie diese Zelle unbedingt aus.

In [ ]:
final_df.to_csv("02-5_final_spO2.csv")
print("Gespeichert als final_spO2.csv")

# Grafiken

## Aufgabe 6: Verteilung der $SpO_2$-Werte

Wie sind die $SpO_2$-Werte verteilt? Geben Sie deskriptive Statistiken aus,
1. für **alle** $SpO_2$-Werte zusammen und
2. für die $SpO_2$-Werte **pro Patient\*in und Aufenthalt**.

*Tipp:* `.describe()` und `.groupby([...])` mit zwei Spalten. (Tutorial 02, Abschnitt 3 und 8)

In [ ]:
final_df["spo2"].describe()

In [ ]:
final_df.groupby(["subject_id", "stay_id"])["spo2"].describe()

## Aufgabe 7: Anzahl der Messungen pro Aufenthalt

Die Anzahl der $SpO_2$-Messungen pro Aufenthalt scheint stark zu schwanken (siehe Spalte `count` oben).

Um uns das genauer anzusehen, erstellen wir ein **Balkendiagramm**, das die Anzahl der $SpO_2$-Messungen pro `stay_id` zeigt. Schreiben Sie dazu eine Funktion `plot_spo2_count_per_stay`, die den DataFrame übergeben bekommt, und zeigen Sie das Diagramm an.

*Tipp:* Zählen mit `.groupby("stay_id").size()`, zeichnen mit `ax.bar(...)` (Tutorial 03, Abschnitt 8). Hilfe zu allen Funktionen finden Sie in der [pyplot-Dokumentation](https://matplotlib.org/stable/api/pyplot_summary.html).

In [ ]:
def plot_spo2_count_per_stay(df):
    anzahl = df.groupby("stay_id").size().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(anzahl.index.astype(str), anzahl.values, color="steelblue")
    ax.set_xlabel("Stay-ID")
    ax.set_ylabel("Anzahl $SpO_2$-Messungen")
    ax.set_title(f"$SpO_2$-Messungen pro Aufenthalt (n = {len(anzahl)} Aufenthalte)")
    ax.tick_params(axis="x", labelrotation=90, labelsize=6)
    fig.tight_layout()
    plt.show()


plot_spo2_count_per_stay(final_df)

Die Aufenthalte sind nach der Anzahl der Messungen sortiert. So sieht man sofort: Einige wenige Aufenthalte haben Hunderte Messungen, viele nur eine Handvoll. Für das spätere Modell ist das wichtig, denn Aufenthalte mit vielen Messungen könnten sonst das Ergebnis dominieren.

## Bonusaufgabe (optional)

Wir wollen genauer betrachten, wie die $SpO_2$-Werte bei einzelnen Patient\*innen aussehen.

1. Wählen Sie zufällig 5 Patient\*innen aus und stellen Sie deren $SpO_2$-Werte als **Boxplots** dar (ohne nach Aufenthalten zu unterscheiden). Für eine Zufallsauswahl können Sie `.sample()` verwenden.
2. Stellen Sie anschließend den **$SpO_2$-Verlauf** eines zufälligen Aufenthalts dar. Dazu wählen Sie zufällig eine Kombination aus `subject_id` und `stay_id` aus und filtern den DataFrame entsprechend.

*Tipp:* Damit jede `subject_id` nur einmal zur Auswahl steht, zuerst `.drop_duplicates()` verwenden. Boxplots mehrerer Gruppen: Tutorial 03, Abschnitt 7. Für lesbare Datumsangaben auf der x-Achse: `ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m. %H:%M"))`.

In [ ]:
# 1. Boxplots für 5 zufällige Patient*innen
patienten = final_df["subject_id"].drop_duplicates().sample(5)

werte = [final_df.loc[final_df["subject_id"] == p, "spo2"] for p in patienten]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(werte)
ax.set_xticks(range(1, 6))
ax.set_xticklabels(patienten.astype(str))
ax.set_xlabel("Subject-ID")
ax.set_ylabel("$SpO_2$ (%)")
ax.set_title("$SpO_2$-Verteilung von 5 zufälligen Patient*innen")
plt.show()

In [ ]:
# 2. SpO2-Verlauf eines zufälligen Aufenthalts
auswahl = final_df[["subject_id", "stay_id"]].drop_duplicates().sample(1)
subject = auswahl["subject_id"].iloc[0]
stay = auswahl["stay_id"].iloc[0]

verlauf = final_df[(final_df["subject_id"] == subject) & (final_df["stay_id"] == stay)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(verlauf["time"], verlauf["spo2"], marker="o", markersize=3)
ax.axhline(90, color="red", linestyle="--", label="90 %")
ax.set_xlabel("Zeit")
ax.set_ylabel("$SpO_2$ (%)")
ax.set_title(f"$SpO_2$-Verlauf – Patient*in {subject}, Aufenthalt {stay}")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m. %H:%M"))
fig.autofmt_xdate()
ax.legend()
plt.show()

Führen Sie die beiden Zellen mehrmals aus: Bei jedem Durchlauf werden andere Patient\*innen gezogen. Wer immer dieselbe Auswahl möchte (z. B. für einen Bericht), gibt `.sample(..., random_state=42)` an.